In [1]:
import os
import re
from pathlib import Path

import pandas as pd


# =========================
# パス
# =========================
BASE = Path(os.environ.get("OneDrive", "")) / "タブレット用"
DB_PATH = BASE / "recipe_db.xlsx"
SHEET_NAME = "Recipes"

OUT_XLSX = BASE / "dup_by_recipe_name.xlsx"


# =========================
# 正規化
# =========================
def norm_name(x: str) -> str:
    """レシピ名の比較用正規化"""
    if x is None:
        return ""
    s = str(x).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


# =========================
# メイン
# =========================
def main():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"DBが見つかりません: {DB_PATH}")

    df = pd.read_excel(DB_PATH, sheet_name=SHEET_NAME)

    # 必須列
    for c in ["RecipeID", "レシピ名"]:
        if c not in df.columns:
            raise ValueError(f"{SHEET_NAME} に列 {c} がありません")

    # 正規化キー
    df["RecipeID"] = df["RecipeID"].astype(str).str.strip()
    df["name_key"] = df["レシピ名"].map(norm_name)

    # ★ レシピ名が同じものを抽出
    dup = df[df.duplicated(subset=["name_key"], keep=False)].copy()

    dup = dup.sort_values(["name_key", "RecipeID"])

    # 出力
    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
        df[["RecipeID", "レシピ名", "name_key"]].to_excel(
            w, sheet_name="DB_KEY", index=False
        )

        if dup.empty:
            pd.DataFrame(
                [{"message": "レシピ名が完全一致する重複はありません"}]
            ).to_excel(w, sheet_name="DUPLICATES", index=False)
        else:
            dup[["RecipeID", "レシピ名"]].to_excel(
                w, sheet_name="DUPLICATES", index=False
            )

    print(f"Saved: {OUT_XLSX}")
    print(f"Duplicate rows: {len(dup)}")


if __name__ == "__main__":
    main()


Saved: C:\Users\spax2\OneDrive\タブレット用\dup_by_recipe_name.xlsx
Duplicate rows: 20
